## ML Forecasting: Prophet
Trains one Prophet model per city on historical `weather_curated` data,
tracks each run in MLflow, and writes 24-hour-ahead forecasts to
`weather_forecast`. Runs as its own pipeline stage, independent of the
raw→processed and processed→gold notebooks.

In [0]:
%pip install prophet
dbutils.library.restartPython()

In [0]:
%run ./00_setup_config

In [0]:
df_dims_city = spark.table("internship_databricks_ws.default.weather_dims_city")
cities = [row["city"] for row in df_dims_city.select("city").distinct().collect()]
print(f"Training forecasts for {len(cities)} cities: {cities}")

In [0]:
import mlflow
import mlflow.prophet
from prophet import Prophet
import pandas as pd

df_curated = spark.table("internship_databricks_ws.default.weather_curated")

forecast_results = []

mlflow.set_experiment("/Shared/weather_forecasting")

for city in cities:

    city_pd = (
        df_curated
        .filter(df_curated.city == city)
        .select("weather_time_pkt", "temperature_c")
        .orderBy("weather_time_pkt")
        .toPandas()
    )

    # Require enough historical observations
    if len(city_pd) < 48:
        print(
            f"Skipping {city} — not enough historical data "
            f"({len(city_pd)} rows, need at least 48)"
        )
        continue

    city_pd = city_pd.rename(
        columns={
            "weather_time_pkt": "ds",
            "temperature_c": "y"
        }
    )

    city_pd["ds"] = pd.to_datetime(city_pd["ds"])
    city_pd["y"] = pd.to_numeric(city_pd["y"], errors="coerce")

    # Remove invalid rows
    city_pd = city_pd.dropna(subset=["ds", "y"])

    with mlflow.start_run(run_name=f"forecast_{city}"):

        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=False
        )

        model.fit(city_pd)

        future = model.make_future_dataframe(
            periods=24,
            freq="h"
        )

        forecast = model.predict(future)

        mlflow.log_param("city", city)
        mlflow.log_param("training_rows", len(city_pd))

        mlflow.prophet.log_model(
            model,
            name="model"
        )

        next_24h = forecast[
            forecast["ds"] > city_pd["ds"].max()
        ][
            ["ds", "yhat", "yhat_lower", "yhat_upper"]
        ].copy()

        next_24h["city"] = city

        forecast_results.append(next_24h)

    print(f"Trained and forecasted: {city}")

In [0]:
if not forecast_results:
    raise ValueError("No cities had enough historical data to generate forecasts.")

from pyspark.sql.functions import current_timestamp

df_all_forecasts = pd.concat(
    forecast_results,
    ignore_index=True
)

df_forecast_spark = (
    spark.createDataFrame(df_all_forecasts)
    .withColumnRenamed("ds", "forecast_time")
    .withColumnRenamed("yhat", "predicted_temp_c")
    .withColumnRenamed("yhat_lower", "predicted_temp_lower")
    .withColumnRenamed("yhat_upper", "predicted_temp_upper")
    .withColumn("generated_at", current_timestamp())
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_forecast
USING DELTA
LOCATION '{gold_path}weather_forecast/'
""")

df_forecast_spark.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "internship_databricks_ws.default.weather_forecast"
    )

print(
    f"Wrote {df_forecast_spark.count()} forecast rows "
    "to weather_forecast"
)

In [0]:
spark.sql("""
SELECT
    city,
    forecast_time,
    predicted_temp_c,
    predicted_temp_lower,
    predicted_temp_upper
FROM internship_databricks_ws.default.weather_forecast
ORDER BY city, forecast_time
LIMIT 30
""").show(truncate=False)